# ЛР 2

## Импорты

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from string import punctuation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 3

## Загрузка данных

In [5]:
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = 'positive'

negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = 'negative'

df = pd.concat([positive, negative], ignore_index=True)

In [6]:
print("Размер выборки:", df.shape)
print("\nПримеры твитов:")
print(df.sample(5, random_state=RANDOM_STATE))

Размер выборки: (226834, 2)

Примеры твитов:
                                                     text     label
46226   пришла с больницы , сказали что все оч плохо :...  positive
3705    @a_moiseeva не, ты чего, с такими деньгами вря...  positive
35408   http://t.co/1DK3rfd9qy и здесь такая живая))) ...  positive
27038   RT @MyChemland: @SugarPony23 \n#СиськаNeverDie...  positive
119655  @aliiina_sh жалко( надеюсь,вы не стали реже об...  negative


In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    df.text, 
    df.label, 
    random_state=RANDOM_STATE
)

print(f"Размер обучающей выборки: {len(x_train)}")
print(f"Размер тестовой выборки: {len(x_test)}")

Размер обучающей выборки: 170125
Размер тестовой выборки: 56709


## 1) Исследование n-грамм

In [9]:
n_values = [1, 2, 3, 4, 5]
results_n = {}

for n in n_values:
    print(f"\n=== n-граммы: 1..{n} ===")
    vectorizer = TfidfVectorizer(
        ngram_range=(1, n),
        max_features=100000,
        min_df=3
    )
    
    X_train_vec = vectorizer.fit_transform(x_train)
    X_test_vec = vectorizer.transform(x_test)
    
    clf = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    clf.fit(X_train_vec, y_train)
    
    pred = clf.predict(X_test_vec)
    report = classification_report(y_test, pred, output_dict=True)
    results_n[n] = report

    print(f"Accuracy: {report['accuracy']:.3f}")
    print(f"F1 (negative): {report['negative']['f1-score']:.3f}")
    print(f"F1 (positive): {report['positive']['f1-score']:.3f}")


=== n-граммы: 1..1 ===
Accuracy: 0.757
F1 (negative): 0.749
F1 (positive): 0.765

=== n-граммы: 1..2 ===
Accuracy: 0.768
F1 (negative): 0.761
F1 (positive): 0.775

=== n-граммы: 1..3 ===
Accuracy: 0.767
F1 (negative): 0.760
F1 (positive): 0.773

=== n-граммы: 1..4 ===
Accuracy: 0.768
F1 (negative): 0.761
F1 (positive): 0.774

=== n-граммы: 1..5 ===
Accuracy: 0.767
F1 (negative): 0.760
F1 (positive): 0.774


### Вывод:
Добавление биграмм заметно улучшает качество. Расширение до 3–5-грамм не улучшает результат и даже немного снижает его. Оптимальный диапазон — биграммы.

## 2) Сравнение с другой моделью

In [12]:
vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=100000,
        min_df=3
    )
X_train_vec = vectorizer.fit_transform(x_train)
X_test_vec = vectorizer.transform(x_test)

lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr.fit(X_train_vec, y_train)
pred_lr = lr.predict(X_test_vec)
print("=== LogisticRegression ===")
print(classification_report(y_test, pred_lr))

svm = LinearSVC(random_state=RANDOM_STATE, max_iter=1000)
svm.fit(X_train_vec, y_train)
pred_svm = svm.predict(X_test_vec)
print("\n=== LinearSVC ===")
print(classification_report(y_test, pred_svm))

=== LogisticRegression ===
              precision    recall  f1-score   support

    negative       0.77      0.75      0.76     27900
    positive       0.76      0.79      0.77     28809

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709


=== LinearSVC ===
              precision    recall  f1-score   support

    negative       0.76      0.75      0.75     27900
    positive       0.76      0.77      0.76     28809

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



Вывод:
Логистическая регрессия показала немного лучшие результаты (accuracy 0.77), чем LinearSVC (0.76).

## 3) Влияние стоп-слов

In [15]:
import nltk
#nltk.download('punkt')
#nltk.download('punkt_tab')
#nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from string import punctuation

In [16]:
stop_words_none = None
stop_words_basic = stopwords.words('russian') + stopwords.words('english')
twitter_noise = ['rt', 'http', 'https', 't.co', 'amp', 'pic', 'twitter', 'com', 'ru']
neutral_words = ['это', 'который', 'свой', 'весь', 'сам', 'такой', 'тоже', 'также', 'или', 'либо', 'потом', 'затем', 'там', 'тут', 'здесь', 'везде', 'всегда', 'никогда', 'иногда']
stop_words_extended = stop_words_basic + list(punctuation) + twitter_noise

variants = {
    'без стоп-слов': stop_words_none,
    'только шумовые слова': twitter_noise,
    'шумовые + нейтральные': twitter_noise + neutral_words,
    'базовые стоп-слова (nltk)': stop_words_basic,
    'расширенные стоп-слова + пунктуация + шум': stop_words_extended
}

ngram = (1, 2)

for name, stop_words in variants.items():
    print(f"\n=== {name} ===")
    vectorizer = TfidfVectorizer(
        ngram_range=ngram,
        stop_words=stop_words,
        lowercase=True,
        max_features=100000,
        min_df=3
    )
    X_train_vec = vectorizer.fit_transform(x_train)
    X_test_vec = vectorizer.transform(x_test)
    
    clf = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    clf.fit(X_train_vec, y_train)
    pred = clf.predict(X_test_vec)
    print(classification_report(y_test, pred))


=== без стоп-слов ===
              precision    recall  f1-score   support

    negative       0.77      0.75      0.76     27900
    positive       0.76      0.79      0.77     28809

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709


=== только шумовые слова ===
              precision    recall  f1-score   support

    negative       0.77      0.75      0.76     27900
    positive       0.76      0.79      0.78     28809

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709


=== шумовые + нейтральные ===
              precision    recall  f1-score   support

    negative       0.77      0.75      0.76     27900
    positive       0.76      0.78      0.77     28809

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77 

Вывод:
Удаление только шумовых токенов практически не изменило качество (accuracy осталась 0.77), а добавление нейтральных общеупотребительных слов не дало улучшения. Использование стандартных стоп-слов из NLTK, как и их расширение пунктуацией, привело к заметному падению accuracy до 0.75 из-за удаления значимых для тональности слов (отрицаний, усилителей). Оптимальная стратегия для данной задачи — либо вовсе не применять стоп-слова, либо ограничиться минимальным списком заведомо неинформативных элементов.

## Вывод:
Наилучшее качество классификации твитов по тональности достигается при использовании биграмм (1–2) и логистической регрессии. Применение стандартных стоп-слов из NLTK ухудшает результат, так как удаляются значимые для тональности слова.
